In [27]:
import pandas as pd
import numpy as np

pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", "{:.2f}".format)

In [28]:
df = pd.read_csv("../data/processed_data/raw_dataset_copy.csv")

print("Dataset Loaded Successfully!")

Dataset Loaded Successfully!


In [29]:
df.head()

,id,age,gender,height,weight,ap_hi,ap_lo,cholesterol,gluc,smoke,alco,active,cardio
0,0,18393,2,168,62.00,110,80,1,1,0,0,1,0
1,1,20228,1,156,85.00,140,90,3,1,0,0,1,1
2,2,18857,1,165,64.00,130,70,3,1,0,0,0,1
3,3,17623,2,169,82.00,150,100,1,1,0,0,1,1
4,4,17474,1,156,56.00,100,60,1,1,0,0,0,0


In [30]:
print("Dataset Shape :", df.shape)

Dataset Shape : (70000, 13)


In [31]:
print("Missing Values\n")
print(df.isnull().sum())

Missing Values

id             0
age            0
gender         0
height         0
weight         0
ap_hi          0
ap_lo          0
cholesterol    0
gluc           0
smoke          0
alco           0
active         0
cardio         0
dtype: int64


In [32]:
duplicates = df.duplicated().sum()

print("Duplicate Rows Before Removing :", duplicates)

df = df.drop_duplicates()

print("Duplicate Rows After Removing :", df.duplicated().sum())

print("New Shape :", df.shape)

Duplicate Rows Before Removing : 0
Duplicate Rows After Removing : 0
New Shape : (70000, 13)


In [33]:
df["age"] = (df["age"] / 365).round(1)

print("Age converted from Days to Years.")

Age converted from Days to Years.


In [34]:
df.rename(columns={
    "ap_hi":"Systolic_BP",
    "ap_lo":"Diastolic_BP",
    "gluc":"Glucose",
    "alco":"Alcohol",
    "active":"Physical_Activity",
    "cardio":"Heart_Disease"
}, inplace=True)

In [35]:
df.dtypes

id                     int64
age                  float64
gender                 int64
height                 int64
weight               float64
Systolic_BP            int64
Diastolic_BP           int64
cholesterol            int64
Glucose                int64
smoke                  int64
Alcohol                int64
Physical_Activity      int64
Heart_Disease          int64
dtype: object

In [36]:
print("Invalid Systolic BP :", (df["Systolic_BP"] <= 0).sum())
print("Invalid Diastolic BP :", (df["Diastolic_BP"] <= 0).sum())

Invalid Systolic BP : 7
Invalid Diastolic BP : 22


In [37]:
df = df[
    (df["Systolic_BP"] > 0) &
    (df["Diastolic_BP"] > 0)
]

print("Shape After Removing Invalid BP :", df.shape)

Shape After Removing Invalid BP : (69971, 13)


In [38]:
print("Invalid Height :", (df["height"] <= 0).sum())
print("Invalid Weight :", (df["weight"] <= 0).sum())

Invalid Height : 0
Invalid Weight : 0


In [39]:
df = df[
    (df["height"] > 0) &
    (df["weight"] > 0)
]

print("Shape :", df.shape)

Shape : (69971, 13)


In [40]:
df["Height_m"] = df["height"] / 100

df["BMI"] = df["weight"] / (df["Height_m"]**2)

df["BMI"] = df["BMI"].round(2)

In [41]:
def bmi_category(bmi):

    if bmi < 18.5:
        return "Underweight"

    elif bmi < 25:
        return "Normal"

    elif bmi < 30:
        return "Overweight"

    else:
        return "Obese"


df["BMI_Category"] = df["BMI"].apply(bmi_category)

In [42]:
def age_group(age):

    if age < 30:
        return "Young"

    elif age < 50:
        return "Middle Age"

    elif age < 65:
        return "Senior"

    else:
        return "Elderly"


df["Age_Group"] = df["age"].apply(age_group)

In [43]:
def bp_category(sys, dia):

    if sys < 120 and dia < 80:
        return "Normal"

    elif sys < 130 and dia < 80:
        return "Elevated"

    elif sys < 140 or dia < 90:
        return "Stage 1"

    else:
        return "Stage 2"


df["BP_Category"] = df.apply(
    lambda row: bp_category(
        row["Systolic_BP"],
        row["Diastolic_BP"]
    ),
    axis=1
)

In [44]:
df["Pulse_Pressure"] = (
    df["Systolic_BP"] -
    df["Diastolic_BP"]
)

In [45]:
df["MAP"] = (
    (
        2 * df["Diastolic_BP"] +
        df["Systolic_BP"]
    ) / 3
)

df["MAP"] = df["MAP"].round(2)

In [46]:
# Remove Medical Outliers
print("="*60)
print("Removing Medical Outliers")
print("="*60)

print("Dataset Shape Before Removing Outliers :", df.shape)

# ----------------------------------------------------
# Age (18 - 100 years)
# ----------------------------------------------------
df = df[
    (df["age"] >= 18) &
    (df["age"] <= 100)
]

# ----------------------------------------------------
# Height (120 cm - 220 cm)
# ----------------------------------------------------
df = df[
    (df["height"] >= 120) &
    (df["height"] <= 220)
]

# ----------------------------------------------------
# Weight (30 kg - 200 kg)
# ----------------------------------------------------
df = df[
    (df["weight"] >= 30) &
    (df["weight"] <= 200)
]

# ----------------------------------------------------
# BMI (15 - 45)
# ----------------------------------------------------
df = df[
    (df["BMI"] >= 15) &
    (df["BMI"] <= 45)
]

# ----------------------------------------------------
# Systolic Blood Pressure (80 - 200)
# ----------------------------------------------------
df = df[
    (df["Systolic_BP"] >= 80) &
    (df["Systolic_BP"] <= 200)
]

# ----------------------------------------------------
# Diastolic Blood Pressure (50 - 130)
# ----------------------------------------------------
df = df[
    (df["Diastolic_BP"] >= 50) &
    (df["Diastolic_BP"] <= 130)
]

print("\nDataset Shape After Removing Outliers :", df.shape)

print("\nRows Removed :", 69971 - len(df))

Removing Medical Outliers
Dataset Shape Before Removing Outliers : (69971, 20)

Dataset Shape After Removing Outliers : (68036, 20)

Rows Removed : 1935


In [47]:
df.head()

,id,age,gender,height,weight,Systolic_BP,Diastolic_BP,cholesterol,Glucose,smoke,Alcohol,Physical_Activity,Heart_Disease,Height_m,BMI,BMI_Category,Age_Group,BP_Category,Pulse_Pressure,MAP
0,0,50.40,2,168,62.00,110,80,1,1,0,0,1,0,1.68,21.97,Normal,Senior,Stage 1,30,90.00
1,1,55.40,1,156,85.00,140,90,3,1,0,0,1,1,1.56,34.93,Obese,Senior,Stage 2,50,106.67
2,2,51.70,1,165,64.00,130,70,3,1,0,0,0,1,1.65,23.51,Normal,Senior,Stage 1,60,90.00
3,3,48.30,2,169,82.00,150,100,1,1,0,0,1,1,1.69,28.71,Overweight,Middle Age,Stage 2,50,116.67
4,4,47.90,1,156,56.00,100,60,1,1,0,0,0,0,1.56,23.01,Normal,Middle Age,Normal,40,73.33


In [48]:
print(df.shape)

(68036, 20)


In [49]:
df.describe()

,id,age,gender,height,weight,Systolic_BP,Diastolic_BP,cholesterol,Glucose,smoke,Alcohol,Physical_Activity,Heart_Disease,Height_m,BMI,Pulse_Pressure,MAP
count,68036.00,68036.00,68036.00,68036.00,68036.00,68036.00,68036.00,68036.00,68036.00,68036.00,68036.00,68036.00,68036.00,68036.00,68036.00,68036.00,68036.00
mean,49984.45,53.32,1.35,164.47,73.74,126.47,81.27,1.36,1.22,0.09,0.05,0.80,0.49,1.64,27.28,45.21,96.33
std,28848.73,6.76,0.48,7.83,13.53,16.45,9.35,0.68,0.57,0.28,0.22,0.40,0.50,0.08,4.85,11.72,10.86
min,0.00,29.60,1.00,120.00,30.00,80.00,50.00,1.00,1.00,0.00,0.00,0.00,0.00,1.20,15.01,-50.00,60.00
25%,25011.50,48.40,1.00,159.00,65.00,120.00,80.00,1.00,1.00,0.00,0.00,1.00,0.00,1.59,23.88,40.00,93.33
50%,50030.50,54.00,1.00,165.00,72.00,120.00,80.00,1.00,1.00,0.00,0.00,1.00,0.00,1.65,26.30,40.00,93.33
75%,74885.25,58.40,2.00,170.00,82.00,140.00,90.00,1.00,1.00,0.00,0.00,1.00,1.00,1.70,30.09,50.00,103.33
max,99999.00,65.00,2.00,207.00,149.00,200.00,130.00,3.00,3.00,1.00,1.00,1.00,1.00,2.07,45.00,135.00,153.33


In [50]:
print(df.columns)

Index(['id', 'age', 'gender', 'height', 'weight', 'Systolic_BP',
       'Diastolic_BP', 'cholesterol', 'Glucose', 'smoke', 'Alcohol',
       'Physical_Activity', 'Heart_Disease', 'Height_m', 'BMI', 'BMI_Category',
       'Age_Group', 'BP_Category', 'Pulse_Pressure', 'MAP'],
      dtype='str')


In [51]:
df.to_csv(
    "../data/processed_data/cleaned_dataset.csv",
    index=False
)

print("Cleaned Dataset Saved Successfully!")

Cleaned Dataset Saved Successfully!


In [52]:
print("="*50)

print("DATA PREPROCESSING COMPLETED")

print("="*50)

print("Final Dataset Shape :", df.shape)

print("Missing Values :", df.isnull().sum().sum())

print("Duplicate Rows :", df.duplicated().sum())

print("="*50)

DATA PREPROCESSING COMPLETED
Final Dataset Shape : (68036, 20)
Missing Values : 0
Duplicate Rows : 0
